# 3D animation: landing on the vehicle that follows a curve

Reads `platform3d_flight.csv`, written by `EINT_PLATFORM3D_eval.m` (flight on the "real" drone `UAV_model.slx`), and draws in 3D
the vehicle (box + orange platform on top, oriented along its path) and the drone (4 arms / 4 rotors, tilted by its real angles
φ, θ, ψ). The animation is shown here and saved as a GIF for the demo / the slides.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import HTML

# Flight written by EINT_PLATFORM3D_eval.m -> change the path if needed
FLIGHT_CSV = r'C:\Users\ezabl\Downloads\Student_folder\Student_folder\rl_interface\platform3d_flight.csv'
GIF = Path('../results/figures/platform3d_landing.gif')

# Columns: t z vz az_prev x vx theta wy ax az y vy phi psi wx wz u1 u2 u3 u4 x_car y_car
F = np.loadtxt(FLIGHT_CSV, delimiter=',')
t, z, x, y = F[:, 0], F[:, 1], F[:, 4], F[:, 10]
wy, ax, az, phi, psi = F[:, 7], F[:, 8], F[:, 9], F[:, 12], F[:, 13]
x_car, y_car = F[:, 20], F[:, 21]
Ts, G = t[1] - t[0], 9.81

# Pitch theta, only for the drawing: the measured theta (column 6) is unusable (huge noise in UAV_model.slx),
# so same complementary filter as platform3d_controller.m (gyroscope integrated + accelerometer). phi and psi are fine.
theta = np.zeros_like(t)
for k in range(1, len(t)):
    theta[k] = theta[k - 1] + Ts * (wy[k] + wy[k - 1]) / 2
    if az[k] + G > 2:
        theta[k] = 0.98 * theta[k] + 0.02 * np.arctan2(ax[k], az[k] + G)

# Heading of the vehicle (direction of its speed), 0 while it is at rest
vxc, vyc = np.gradient(x_car, Ts), np.gradient(y_car, Ts)
heading = np.where(np.hypot(vxc, vyc) > 1e-3, np.arctan2(vyc, vxc), 0.0)
gap = np.hypot(x - x_car, y - y_car)
print(f'{len(t)} samples, {t[-1]:.1f} s; final distance drone - platform: {gap[-1]:.3f} m')

**Drawing** (sizes in m, chosen for readability): ground at z = −0.25 m; the vehicle is a box whose top (z = 0, in orange)
is the platform, centred on the position of the vehicle and turned along its heading. The drone is a cross of 2 arms
(motors 1-3 on its x axis, 2-4 on its y axis) turned by the rotation R = Rz(ψ)·Ry(θ)·Rx(φ), the same convention as the
thrust direction in the model.

In [ ]:
Z_GROUND, ARM = -0.25, 0.15                          # ground height, half-length of a drone arm (m)
CAR_L, CAR_W, PLAT = 0.9, 0.5, 0.4                     # vehicle length, width, side of the square platform (m)

def rot_z(a):
    c, s = np.cos(a), np.sin(a)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def rotation(ph, th, ps):                              # body -> world, ZYX (yaw, pitch, roll)
    cp, sp, ct, st = np.cos(ph), np.sin(ph), np.cos(th), np.sin(th)
    Ry = np.array([[ct, 0, st], [0, 1, 0], [-st, 0, ct]])
    Rx = np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]])
    return rot_z(ps) @ Ry @ Rx

def box_faces(xc, yc, h):
    # 6 faces of the vehicle body: from the ground to z = -0.02 (just under the platform), turned by the heading h
    corners = np.array([[sx * CAR_L / 2, sy * CAR_W / 2] for sx, sy in [(-1, -1), (1, -1), (1, 1), (-1, 1)]])
    corners = corners @ rot_z(h)[:2, :2].T + [xc, yc]
    low = [(*c, Z_GROUND + 0.05) for c in corners]; high = [(*c, -0.02) for c in corners]
    sides = [[low[i], low[(i + 1) % 4], high[(i + 1) % 4], high[i]] for i in range(4)]
    return [low, high] + sides

def platform_face(xc, yc, h):
    sq = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]]) * PLAT / 2 @ rot_z(h)[:2, :2].T + [xc, yc]
    return [[(*c, 0.0) for c in sq]]

fig = plt.figure(figsize=(9, 7))
axis = fig.add_subplot(projection='3d', computed_zorder=False)   # draw in the order given by zorder (else the box hides the drone)
pad = 0.6
axis.set_xlim(min(x.min(), x_car.min()) - pad, max(x.max(), x_car.max()) + pad)
axis.set_ylim(min(y.min(), y_car.min()) - pad, max(y.max(), y_car.max()) + pad)
axis.set_zlim(Z_GROUND, 0.9)
axis.set_box_aspect((np.ptp(axis.get_xlim()), np.ptp(axis.get_ylim()), np.ptp(axis.get_zlim())))
axis.set_xlabel('x [m]'); axis.set_ylabel('y [m]'); axis.set_zlabel('z [m]')
axis.view_init(elev=30, azim=-60)

# ground and path of the vehicle (dashed)
gx, gy = np.meshgrid(axis.get_xlim(), axis.get_ylim())
axis.plot_surface(gx, gy, np.full_like(gx, Z_GROUND), color='lightgray', alpha=0.3, zorder=0)
axis.plot(x_car, y_car, np.full_like(t, Z_GROUND), '--', color='steelblue', lw=1, zorder=1)

body = Poly3DCollection(box_faces(0, 0, 0), facecolor='lightsteelblue', edgecolor='k', alpha=0.9, zorder=2)
platform = Poly3DCollection(platform_face(0, 0, 0), facecolor='orange', edgecolor='k', zorder=3)
axis.add_collection3d(body); axis.add_collection3d(platform)
arms = [axis.plot([], [], [], 'k-', lw=3, zorder=6)[0] for _ in range(2)]            # drone: 2 arms
rotors = axis.plot([], [], [], 'o', ms=8, mfc='tab:red', mec='k', zorder=7)[0]        # 4 rotors
trail = axis.plot([], [], [], ':', color='tab:red', lw=1.2, zorder=5)[0]                # path of the drone
drop = axis.plot([], [], [], '-', color='tab:red', lw=1, alpha=0.6, zorder=4)[0]        # vertical line drone -> platform level (z = 0)
dot = axis.plot([], [], [], 'x', color='tab:red', zorder=4)[0]                          # point just under the drone at z = 0
label = axis.text2D(0.02, 0.95, '', transform=axis.transAxes, family='monospace')

ARMS_BODY = np.array([[-ARM, 0, 0], [0, ARM, 0], [ARM, 0, 0], [0, -ARM, 0]])   # motors 1, 2, 3, 4 in the drone frame

def draw(k):
    body.set_verts(box_faces(x_car[k], y_car[k], heading[k]))
    platform.set_verts(platform_face(x_car[k], y_car[k], heading[k]))
    M = ARMS_BODY @ rotation(phi[k], theta[k], psi[k]).T + [x[k], y[k], z[k]]    # motors in the world frame
    for a, (i, j) in zip(arms, [(0, 2), (1, 3)]):
        a.set_data_3d(M[[i, j], 0], M[[i, j], 1], M[[i, j], 2])
    rotors.set_data_3d(M[:, 0], M[:, 1], M[:, 2])
    trail.set_data_3d(x[:k + 1], y[:k + 1], z[:k + 1])
    drop.set_data_3d([x[k], x[k]], [y[k], y[k]], [z[k], 0.0])   # shows where the drone is above the platform (3D perspective is misleading)
    dot.set_data_3d([x[k]], [y[k]], [0.0])
    label.set_text(f't = {t[k]:4.2f} s   height {z[k]:+.2f} m   distance to platform {gap[k]:.2f} m')
    return []

anim = animation.FuncAnimation(fig, draw, frames=len(t), interval=Ts * 1000)
GIF.parent.mkdir(parents=True, exist_ok=True)
anim.save(GIF, writer=animation.PillowWriter(fps=int(round(1 / Ts))))   # real time: 20 images per second
plt.close(fig)
print('saved:', GIF.resolve())
HTML(anim.to_jshtml())